# Elección del modelo: Qwen 2.5 3B-Instruct

**¿Por qué Qwen 2.5 3B?**
- Ya está en producción en NormaBot via Ollama para la etapa de grading
- Open-weight, totalmente compatible con LoRA y QLoRA
- Suficientemente ligero para entrenarlo en una GPU T4 (Colab) o A10G
- Muy buena capacidad semántica en español para su tamaño
- La tarea es clasificación binaria simple, no requiere un modelo más grande

**Parámetros:** ~3.000 millones  
**Cuantización durante entrenamiento:** 4-bit NF4 (QLoRA, `bitsandbytes`)  
**Adaptadores LoRA:** r=8, alpha=16, capas de atención (`q_proj`, `k_proj`, `v_proj`, `o_proj`)

> Este notebook está diseñado para ejecutarse en **Google Colab con GPU T4**
> o en cualquier entorno con GPU y ≥ 8 GB de VRAM.


# Instalación de librerías y comprobación de entorno


In [ ]:
%%capture
# Desinstalar versiones conflictivas si las hay
!pip uninstall -y torch torchvision torchaudio bitsandbytes transformers peft trl

# PyTorch con soporte CUDA 12.1 (compatible con Colab T4 y A10G)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Stack de fine-tuning
!pip install \
    transformers==4.46.3 \
    tokenizers \
    accelerate \
    peft==0.13.2 \
    trl==0.12.1 \
    bitsandbytes==0.44.1 \
    datasets

# Métricas y tracking
!pip install scikit-learn mlflow python-dotenv pandas

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# En Colab: subir functions.py a /content/ junto con grading_dataset.jsonl
sys.path.insert(0, "/content")
from functions import (
    LABEL_RELEVANTE, LABEL_NO_RELEVANTE, LABELS, GRADING_SYSTEM_PROMPT,
    check_gpu, load_grading_dataset, split_dataset, show_split_stats,
    build_grading_messages, format_training_prompt, examples_to_hf_dataset,
    load_tokenizer, load_model_4bit, build_peft_model, get_training_args,
    run_training, save_adapter, predict_relevance, evaluate_model,
    print_comparison, log_experiment,
)

check_gpu()

In [ ]:
# ============================================================
# MONTAR GOOGLE DRIVE (para guardar el modelo de forma persistente)
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# CONFIGURACIÓN DE RUTAS Y CONSTANTES
# ============================================================

# Dataset — copiar grading_dataset.jsonl al root de Colab antes de ejecutar
DATASET_PATH = Path("/content/grading_dataset.jsonl")

LABEL_RELEVANTE    = "relevante"
LABEL_NO_RELEVANTE = "no relevante"
LABELS = [LABEL_RELEVANTE, LABEL_NO_RELEVANTE]

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Salida en Google Drive — persiste entre sesiones de Colab
DRIVE_BASE   = Path("/content/drive/MyDrive/normabot")
OUTPUT_DIR   = str(DRIVE_BASE / "qwen-grader-lora")
ADAPTER_PATH = str(DRIVE_BASE / "qwen-grader-lora" / "adapter_final")
MERGED_PATH  = str(DRIVE_BASE / "qwen-grader-merged")   # solo si se hace merge

MAX_SEQ_LEN = 512

print("Configuración:")
print(f"  Modelo base:    {MODEL_ID}")
print(f"  Dataset:        {DATASET_PATH}  (existe: {DATASET_PATH.exists()})")
print(f"  Output dir:     {OUTPUT_DIR}")
print(f"  Adapter path:   {ADAPTER_PATH}")
print(f"  Max seq len:    {MAX_SEQ_LEN}")

In [ ]:
all_data = load_grading_dataset(DATASET_PATH)

In [ ]:
train_data, val_data, test_data = split_dataset(all_data)

In [ ]:
tokenizer  = load_tokenizer(MODEL_ID)
base_model = load_model_4bit(MODEL_ID)

# Formato del prompt de entrenamiento

Para el fine-tuning usamos el **chat template nativo de Qwen 2.5** (formato `<|im_start|>`).
Cada ejemplo de entrenamiento tiene la forma:

```
<|im_start|>system
Eres un asistente especializado en normativa de IA...<|im_end|>
<|im_start|>user
Consulta: {query}

Documento: {document}

¿Es este documento relevante para responder la consulta?<|im_end|>
<|im_start|>assistant
{label}<|im_end|>
```

El modelo aprende a generar directamente `relevante` o `no relevante` como turno de assistant.


In [ ]:
sample_formatted = format_training_prompt(train_data[0], tokenizer)
print("Ejemplo de prompt de entrenamiento:")
print("-" * 65)
print(sample_formatted)
print("-" * 65)
print(f"Longitud: {len(sample_formatted)} chars")

In [ ]:
train_dataset = examples_to_hf_dataset(train_data, tokenizer)
val_dataset   = examples_to_hf_dataset(val_data,   tokenizer)

print(f"Dataset de entrenamiento: {len(train_dataset)} ejemplos")
print(f"Dataset de validacion:    {len(val_dataset)} ejemplos")

# Fine-tuning con QLoRA

**QLoRA** (Quantized Low-Rank Adaptation) combina:
- **Cuantización 4-bit NF4** para reducir el uso de VRAM
- **LoRA** para entrenar solo un subconjunto pequeño de parámetros

Con r=8 entrenamos aproximadamente **el 1-2% de los parámetros totales** del modelo,
lo que hace el fine-tuning viable en una GPU T4 (16 GB VRAM).

Usamos `SFTTrainer` de `trl` (Supervised Fine-Tuning Trainer), que gestiona automáticamente
el packing de secuencias, la máscara de padding y la pérdida solo sobre el turno de assistant.


In [ ]:
try:
    del base_model
    torch.cuda.empty_cache()
    print("Modelo baseline liberado de VRAM.")
except NameError:
    pass

model = load_model_4bit(MODEL_ID, for_training=True)

In [ ]:
model, lora_config = build_peft_model(model)

In [ ]:
training_args = get_training_args(OUTPUT_DIR)

In [ ]:
train_result = run_training(
    model, training_args, train_dataset, val_dataset, tokenizer, MAX_SEQ_LEN
)

# Guardado del modelo

Guardamos el **adaptador LoRA** en Google Drive (`MyDrive/normabot/qwen-grader-lora/`).
El adaptador pesa solo unos pocos MB y es suficiente para inferencia si se carga junto al modelo base.

Para producción con Ollama, descomenta el bloque de merge para obtener el modelo completo
y convertirlo a GGUF.


In [ ]:
metadata = {
    "model_id":              MODEL_ID,
    "task":                  "rag_relevance_grading",
    "labels":                LABELS,
    "lora_r":                lora_config.r,
    "lora_alpha":            lora_config.lora_alpha,
    "train_size":            len(train_data),
    "val_size":              len(val_data),
    "test_size":             len(test_data),
    "grading_system_prompt": GRADING_SYSTEM_PROMPT,
}
save_adapter(model, tokenizer, ADAPTER_PATH, metadata)

# OPCIONAL: merge adaptador + modelo base para convertir a GGUF
# os.makedirs(MERGED_PATH, exist_ok=True)
# from peft import PeftModel
# base_for_merge = load_model_4bit(MODEL_ID)
# merged = PeftModel.from_pretrained(base_for_merge, ADAPTER_PATH)
# merged = merged.merge_and_unload()
# save_adapter(merged, tokenizer, MERGED_PATH, metadata)
# print("Siguiente paso: convertir a GGUF con llama.cpp y subir a Ollama.")